<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week8/Day1/IPL_Complex_SQL_Daily_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge — IPL Complex SQL Queries

## Project Overview

Analysis of Indian Premier League (IPL) player statistics
using Python, Pandas, SQLite, and SQL queries.

### Objectives
- Explore the database structure
- Analyze batsmen and runs
- Calculate fifties and hundreds
- Identify best bowling figures
- Build comprehensive player career metrics

In [1]:
import pandas as pd
import sqlite3
import zipfile
from pathlib import Path

# Locate the uploaded ZIP file
zip_files = list(Path("/content").glob("*.zip"))

if not zip_files:
    raise FileNotFoundError("No ZIP file found in /content.")

zip_path = zip_files[0]
extract_path = Path("/content/ipl_dataset")

# Extract the dataset
extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(extract_path)

print("Dataset extracted successfully!")
print("\nExtracted files:")

for path in extract_path.rglob("*"):
    if path.is_file():
        print(path.relative_to(extract_path))

Dataset extracted successfully!

Extracted files:
database.sqlite


## Task 1: Load and Explore the Data

In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

# Locate the SQLite database
database_files = list(
    Path("/content/ipl_dataset").rglob("database.sqlite")
)

if not database_files:
    raise FileNotFoundError("IPL database not found.")

database_path = database_files[0]

# Connect to the database
connection = sqlite3.connect(str(database_path))

print("Connected to IPL database successfully!")

# Get all table names
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

print("\nAvailable tables:")
display(tables)

# Display column names for every table
for table_name in tables["name"]:
    columns = pd.read_sql_query(
        f'PRAGMA table_info("{table_name}");',
        connection
    )

    print(f"\nTable: {table_name}")
    print("Columns:", ", ".join(columns["name"]))

Connected to IPL database successfully!

Available tables:


,name
0,Ball_by_Ball
1,Batsman_Scored
2,Batting_Style
3,Bowling_Style
4,City
5,Country
6,Extra_Runs
7,Extra_Type
8,Match
9,Out_Type



Table: Ball_by_Ball
Columns: Match_Id, Over_Id, Ball_Id, Innings_No, Team_Batting, Team_Bowling, Striker_Batting_Position, Striker, Non_Striker, Bowler

Table: Batsman_Scored
Columns: Match_Id, Over_Id, Ball_Id, Runs_Scored, Innings_No

Table: Batting_Style
Columns: Batting_Id, Batting_hand

Table: Bowling_Style
Columns: Bowling_Id, Bowling_skill

Table: City
Columns: City_Id, City_Name, Country_id

Table: Country
Columns: Country_Id, Country_Name

Table: Extra_Runs
Columns: Match_Id, Over_Id, Ball_Id, Extra_Type_Id, Extra_Runs, Innings_No

Table: Extra_Type
Columns: Extra_Id, Extra_Name

Table: Match
Columns: Match_Id, Team_1, Team_2, Match_Date, Season_Id, Venue_Id, Toss_Winner, Toss_Decide, Win_Type, Win_Margin, Outcome_type, Match_Winner, Man_of_the_Match

Table: Out_Type
Columns: Out_Id, Out_Name

Table: Outcome
Columns: Outcome_Id, Outcome_Type

Table: Player
Columns: Player_Id, Player_Name, DOB, Batting_hand, Bowling_skill, Country_Name

Table: Player_Match
Columns: Match_Id, P

In [3]:
# Explore the Player_Match table

sql = """
SELECT *
FROM Player_Match
LIMIT 10;
"""

df_sql = pd.read_sql_query(sql, connection)

display(df_sql)

,Match_Id,Player_Id,Role_Id,Team_Id
0,335987,1,1,1
1,335987,2,3,1
2,335987,3,3,1
3,335987,4,3,1
4,335987,5,3,1
5,335987,6,1,2
6,335987,7,3,2
7,335987,8,3,2
8,335987,9,3,2
9,335987,10,3,2


### Task 1 — Analysis

The IPL database contains multiple related tables with information about players, matches, teams, and player performances.

The Player_Match table connects each player to a match, role, and team.

These relationships will be used to analyze player statistics using SQL.

## Query 1: Select All Columns from Player_Match

In [4]:
sql = """
SELECT *
FROM Player_Match;
"""

df_sql = pd.read_sql_query(sql, connection)

display(df_sql)

print("Total records:", len(df_sql))

,Match_Id,Player_Id,Role_Id,Team_Id
0,335987,1,1,1
1,335987,2,3,1
2,335987,3,3,1
3,335987,4,3,1
4,335987,5,3,1
...,...,...,...,...
12689,981024,385,3,11
12690,981024,394,3,11
12691,981024,429,3,11
12692,981024,434,3,2


Total records: 12694


### Query 1 — Analysis

The Player_Match table contains 12,694 records and 4 columns.

Each record identifies a player's participation in a match, including the player ID, match ID, role ID, and team ID.

This information helps establish relationships between players, teams, and matches.

## Query 2: Batsman vs Runs

In [5]:
sql = """
SELECT
    p.Player_Id,
    p.Player_Name,
    SUM(bs.Runs_Scored) AS total_runs
FROM Ball_by_Ball AS bb
INNER JOIN Batsman_Scored AS bs
    ON bb.Match_Id = bs.Match_Id
    AND bb.Over_Id = bs.Over_Id
    AND bb.Ball_Id = bs.Ball_Id
    AND bb.Innings_No = bs.Innings_No
INNER JOIN Player AS p
    ON bb.Striker = p.Player_Id
GROUP BY
    p.Player_Id,
    p.Player_Name
ORDER BY total_runs DESC;
"""

df_sql = pd.read_sql_query(sql, connection)

display(df_sql.head(10))

print("Total batsmen:", len(df_sql))

,Player_Id,Player_Name,total_runs
0,21,SK Raina,4106
1,8,V Kohli,4105
2,57,RG Sharma,3874
3,40,G Gambhir,3634
4,162,CH Gayle,3447
5,46,RV Uthappa,3390
6,187,DA Warner,3373
7,20,MS Dhoni,3270
8,110,AB de Villiers,3270
9,42,S Dhawan,3082


Total batsmen: 434


### Query 2 — Analysis

The query identified 434 batsmen in the IPL database.

SK Raina was the leading run scorer with 4,106 runs, followed closely by V Kohli with 4,105 runs.

RG Sharma ranked third with 3,874 runs.

The results highlight the players with the highest total batting runs recorded in the dataset.

## Query 3: Fifties and Hundreds

In [6]:
sql = """
WITH innings_runs AS (
    SELECT
        bb.Striker AS Player_Id,
        bb.Match_Id,
        bb.Innings_No,
        SUM(bs.Runs_Scored) AS runs
    FROM Ball_by_Ball AS bb
    INNER JOIN Batsman_Scored AS bs
        ON bb.Match_Id = bs.Match_Id
        AND bb.Over_Id = bs.Over_Id
        AND bb.Ball_Id = bs.Ball_Id
        AND bb.Innings_No = bs.Innings_No
    GROUP BY
        bb.Striker,
        bb.Match_Id,
        bb.Innings_No
)
SELECT
    p.Player_Id,
    p.Player_Name,
    SUM(CASE
        WHEN ir.runs BETWEEN 50 AND 99 THEN 1
        ELSE 0
    END) AS fifties,
    SUM(CASE
        WHEN ir.runs >= 100 THEN 1
        ELSE 0
    END) AS hundreds
FROM innings_runs AS ir
INNER JOIN Player AS p
    ON ir.Player_Id = p.Player_Id
GROUP BY
    p.Player_Id,
    p.Player_Name
ORDER BY
    hundreds DESC,
    fifties DESC;
"""

df_sql = pd.read_sql_query(sql, connection)

display(df_sql.head(10))

,Player_Id,Player_Name,fifties,hundreds
0,162,CH Gayle,20,5
1,8,V Kohli,26,4
2,110,AB de Villiers,21,3
3,187,DA Warner,32,2
4,41,V Sehwag,16,2
5,32,SR Watson,14,2
6,185,M Vijay,13,2
7,53,AC Gilchrist,11,2
8,2,BB McCullum,11,2
9,57,RG Sharma,29,1


### Query 3 — Analysis

CH Gayle recorded the highest number of hundreds (5), followed by V Kohli (4) and AB de Villiers (3).

DA Warner recorded 32 fifties, the highest number among the top 10 players displayed.

These results highlight the batsmen who achieved the most fifty-run and hundred-run innings in the dataset.

## Query 4: Best Bowling Figures

In [7]:
sql = """
WITH bowler_match_wickets AS (
    SELECT
        bb.Bowler AS Player_Id,
        bb.Match_Id,
        COUNT(*) AS wickets
    FROM Ball_by_Ball AS bb
    INNER JOIN Wicket_Taken AS wt
        ON bb.Match_Id = wt.Match_Id
        AND bb.Over_Id = wt.Over_Id
        AND bb.Ball_Id = wt.Ball_Id
        AND bb.Innings_No = wt.Innings_No
    INNER JOIN Out_Type AS ot
        ON wt.Kind_Out = ot.Out_Id
    WHERE LOWER(ot.Out_Name) IN (
        'caught',
        'bowled',
        'lbw',
        'stumped',
        'caught and bowled',
        'hit wicket'
    )
    GROUP BY
        bb.Bowler,
        bb.Match_Id
),
best_figures AS (
    SELECT
        Player_Id,
        MAX(wickets) AS best_wickets
    FROM bowler_match_wickets
    GROUP BY Player_Id
)
SELECT
    p.Player_Id,
    p.Player_Name,
    bf.best_wickets
FROM best_figures AS bf
INNER JOIN Player AS p
    ON bf.Player_Id = p.Player_Id
ORDER BY
    bf.best_wickets DESC,
    p.Player_Name
LIMIT 10;
"""

df_sql = pd.read_sql_query(sql, connection)

display(df_sql)

,Player_Id,Player_Name,best_wickets
0,430,A Zampa,6
1,102,Sohail Tanvir,6
2,124,A Kumble,5
3,136,A Mishra,5
4,137,AD Mascarenhas,5
5,50,Harbhajan Singh,5
6,84,I Sharma,5
7,346,JD Unadkat,5
8,310,JP Faulkner,5
9,151,L Balaji,5


### Query 4 — Analysis

A Zampa and Sohail Tanvir recorded the best bowling performances, each taking 6 wickets in a single match.

Several other bowlers, including A Kumble, A Mishra, and Harbhajan Singh, achieved 5 wickets in a match.

These results identify the highest single-match wicket totals recorded by bowlers in the IPL dataset.

## Query 5: Comprehensive Career Metrics

In [8]:
sql = """
WITH batting_runs AS (
    SELECT
        bb.Striker AS Player_Id,
        SUM(bs.Runs_Scored) AS total_runs
    FROM Ball_by_Ball AS bb
    INNER JOIN Batsman_Scored AS bs
        ON bb.Match_Id = bs.Match_Id
        AND bb.Over_Id = bs.Over_Id
        AND bb.Ball_Id = bs.Ball_Id
        AND bb.Innings_No = bs.Innings_No
    GROUP BY bb.Striker
),
innings_runs AS (
    SELECT
        bb.Striker AS Player_Id,
        bb.Match_Id,
        bb.Innings_No,
        SUM(bs.Runs_Scored) AS runs
    FROM Ball_by_Ball AS bb
    INNER JOIN Batsman_Scored AS bs
        ON bb.Match_Id = bs.Match_Id
        AND bb.Over_Id = bs.Over_Id
        AND bb.Ball_Id = bs.Ball_Id
        AND bb.Innings_No = bs.Innings_No
    GROUP BY
        bb.Striker,
        bb.Match_Id,
        bb.Innings_No
),
batting_milestones AS (
    SELECT
        Player_Id,
        SUM(CASE WHEN runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
        SUM(CASE WHEN runs >= 100 THEN 1 ELSE 0 END) AS hundreds
    FROM innings_runs
    GROUP BY Player_Id
),
bowler_match_wickets AS (
    SELECT
        bb.Bowler AS Player_Id,
        bb.Match_Id,
        COUNT(*) AS wickets
    FROM Ball_by_Ball AS bb
    INNER JOIN Wicket_Taken AS wt
        ON bb.Match_Id = wt.Match_Id
        AND bb.Over_Id = wt.Over_Id
        AND bb.Ball_Id = wt.Ball_Id
        AND bb.Innings_No = wt.Innings_No
    INNER JOIN Out_Type AS ot
        ON wt.Kind_Out = ot.Out_Id
    WHERE LOWER(ot.Out_Name) IN (
        'caught',
        'bowled',
        'lbw',
        'stumped',
        'caught and bowled',
        'hit wicket'
    )
    GROUP BY
        bb.Bowler,
        bb.Match_Id
),
bowling_stats AS (
    SELECT
        Player_Id,
        SUM(wickets) AS total_wickets,
        MAX(wickets) AS best_wickets
    FROM bowler_match_wickets
    GROUP BY Player_Id
)
SELECT
    p.Player_Id,
    p.Player_Name,
    COUNT(DISTINCT pm.Match_Id) AS matches_played,
    COALESCE(br.total_runs, 0) AS total_runs,
    COALESCE(bm.fifties, 0) AS fifties,
    COALESCE(bm.hundreds, 0) AS hundreds,
    COALESCE(bw.total_wickets, 0) AS total_wickets,
    COALESCE(bw.best_wickets, 0) AS best_wickets
FROM Player AS p
LEFT JOIN Player_Match AS pm
    ON p.Player_Id = pm.Player_Id
LEFT JOIN batting_runs AS br
    ON p.Player_Id = br.Player_Id
LEFT JOIN batting_milestones AS bm
    ON p.Player_Id = bm.Player_Id
LEFT JOIN bowling_stats AS bw
    ON p.Player_Id = bw.Player_Id
GROUP BY
    p.Player_Id,
    p.Player_Name
ORDER BY
    total_runs DESC,
    total_wickets DESC;
"""

df_sql = pd.read_sql_query(sql, connection)

display(df_sql.head(10))

print("Total players:", len(df_sql))

,Player_Id,Player_Name,matches_played,total_runs,fifties,hundreds,total_wickets,best_wickets
0,21,SK Raina,146,4106,28,1,24,2
1,8,V Kohli,138,4105,26,4,4,2
2,57,RG Sharma,142,3874,29,1,15,4
3,40,G Gambhir,132,3634,31,0,0,0
4,162,CH Gayle,91,3447,20,5,18,3
5,46,RV Uthappa,135,3390,17,0,0,0
6,187,DA Warner,100,3373,32,2,0,0
7,20,MS Dhoni,142,3270,16,0,0,0
8,110,AB de Villiers,119,3270,21,3,0,0
9,42,S Dhawan,113,3082,25,0,4,1


Total players: 469


### Query 5 — Analysis

The comprehensive query analyzed 469 IPL players, combining batting and bowling career statistics.

SK Raina was the leading run scorer with 4,106 runs in 146 matches, followed closely by V Kohli with 4,105 runs in 138 matches.

CH Gayle recorded the highest number of hundreds (5) among the top 10 run scorers, while DA Warner achieved the most fifties (32).

RG Sharma demonstrated contributions in both batting and bowling, scoring 3,874 runs and taking 15 wickets.

These results provide a comprehensive overview of player performance by combining matches played, total runs, batting milestones, total wickets, and best single-match bowling performances.